# Setup

In [1]:
# import libraries used in this notebook
from scipy.io import loadmat
import pandas as pd
import numpy as np
import glob
import os
import re
import google.colab.drive as drive
from scipy.stats import norm
import math
import scipy.integrate as integrate
import matplotlib.pyplot as plt
from scipy.stats import f
from scipy.stats import probplot
from scipy.stats import shapiro
from scipy.stats import levene
# !pip install pingouin
# import pingouin as pg

In [2]:
# mount your google drive to this colab runtime
# when prompted, connect to account with access to LC project
mountpoint = '/content/G' # arbitrary mountpoint
drive.mount(mountpoint, force_remount=True)

Mounted at /content/G


In [3]:
# @title Helper Functions

def propogate_fa_cr_stimlev(df, sdt_cts_list, groupby_list, stimlev_var='stimLev', task_var='task', s1_stimlev_var=0):
  """
  Propagates false alarm and correct rejection counts for the grouby-ed dataframe.

  Input:
  - df (pd.DataFrame): The dataframe that has been grouped by groupby_list.
  - sdt_cts_list (list): List of confusion matrix outcomes (e.g., hit, miss, fa, cr).
  - groupby_list (list): Columns used to group df.
  - stimlev_var (str, optional): The name of the stimulus level variable. Defaults to 'stimLev'.
  - task_var (str, optional): The name of the task variable. Defaults to 'task'.
  - s1_stimlev_var (numeric, optional): The stimulus level used for reference (e.g., 0). Defaults to 0.

  Output:
  - df_propagated (pd.DataFrame): Grouped dataframe with fa and cr propagated per groupby_list.
  """
  # reset index
  df = df.reset_index()

  # List columns to keep so there are no duplicated columns (e.g hit_x, hit_y) when merged
  s1_only_cols = [i for i in sdt_cts_list if ('fa' in i) or ('cr' in i)] # (e.g, "noise", 0hz) stimLev=0)
  s2_only_cols = [stimlev_var] + [i for i in sdt_cts_list if ('hit' in i) or ('miss' in i)]  # (e.g "signal", stimLev=8)

  s1_col_keep = df.columns.tolist()
  for col in s2_only_cols:
    s1_col_keep.remove(col)

  s2_col_keep = df.columns.tolist()
  for col in s1_only_cols:
    s2_col_keep.remove(col)

  merge_on = groupby_list.remove(stimlev_var)

  stimlev_list = df[stimlev_var].unique().tolist() # i.e [0, 8, 16, 23, 64, 0.6, 0.12, 0.24, 0.48]
  task_list = df[task_var].unique().tolist() # e.g ['aud', 'vis']

  # copy list and remove s1_stimlev
  s2_stimlev_list = list(stimlev_list)
  s2_stimlev_list.remove(s1_stimlev_var)

  # propogate cr and fa counts for each stimulus level
  concatted = pd.DataFrame() # initialize final DataFrame
  for stimlev in s2_stimlev_list: # omit stimlev==0 in loop
    for task in task_list:
      s1_slice = df.query(f"{stimlev_var}=={s1_stimlev_var} & {task_var}=='{task}'")[[*s1_col_keep]]
      s2_slice = df.query(f"{stimlev_var}=={stimlev} & {task_var}=='{task}'")[[*s2_col_keep]]

      merged = pd.merge(s2_slice, s1_slice, on=groupby_list, suffixes=("", "_s1"))
      concatted = pd.concat([concatted, merged])

  df_propogated = concatted.sort_values(by=[*groupby_list, stimlev_var])
  return df_propogated

def calculate_sdt(hit, miss, fa, cr, edge_correction="loglinear"):
    """ returns a dict with sdt measures: {'d', 'beta', 'c', 'Ad'}

    Edge correction methods:
    - "loglinear" adds 0.5 to both the number of hits and number of false alarms, and adds 1 to number
    of signal and number of noise trials (Hautus, 1995).
    - "halfhit": sets rates of 0 to 0.5/n and rates of 1 to (n-0.5)/n, where n is the total counts of
    hits or false alarms (Macmillan & Kaplan, 1985).

    """
    Z = norm.ppf

    if edge_correction=="loglinear":
        # add 0.5 to hit and fa to avoid d' infinity
        hit_adjusted = hit + 0.5
        fa_adjusted = fa + 0.5
        # add 1 to signal, 1 to noise trials
        signal_trials = (hit + miss + 1)
        noise_trials = (fa + cr + 1)
        # calculate hit_rate and fa_rate
        hit_rate = hit_adjusted / signal_trials
        fa_rate = fa_adjusted / noise_trials
    elif edge_correction=="halfhit":
        # Floors and ceilings are replaced by half hits and half FA's
        half_hit = 0.5 / (hit + miss)
        half_fa = 0.5 / (fa + cr)
        # Calculate hit_rate and avoid d' infinity
        hit_rate = hit / (hit + miss)
        if hit_rate == 1:
            hit_rate = 1 - half_hit
        if hit_rate == 0:
            hit_rate = half_hit
        # Calculate false alarm rate and avoid d' infinity
        fa_rate = fa / (fa + cr)
        if fa_rate == 1:
            fa_rate = 1 - half_fa
        if fa_rate == 0:
            fa_rate = half_fa
    else:
        print("input either 'loglinear' or 'halfhit' (str)")

    # Return d', beta, c and Ad'
    sdt_meas = {}
    sdt_meas['d'] = Z(hit_rate) - Z(fa_rate)
    sdt_meas['beta'] = math.exp((Z(fa_rate)**2 - Z(hit_rate)**2) / 2)
    sdt_meas['c'] = -(Z(hit_rate) + Z(fa_rate)) / 2
    sdt_meas['Ad'] = norm.cdf(sdt_meas['d'] / math.sqrt(2))
    sdt_meas['hit_rate'] = hit / (hit + miss)
    sdt_meas['miss_rate'] = miss / (hit + miss)
    sdt_meas['fa_rate'] = fa / (fa + cr)
    sdt_meas['cr_rate'] = cr / (fa + cr)
    return(sdt_meas)

def calculate_sdt_df(df):
  """
  Calculate Signal Detection Theory (SDT) measures for a grouped DataFrame.

  Given a groupby-ed DataFrame containing hits, misses, false alarms, and correct rejections,
  this function uses the `calculate_sdt` function to calculate SDT measures: d', beta, c, and Ad.

  Parameters:
      df (pd.DataFrame): A groupby-ed DataFrame with columns 'hit', 'miss', 'fa', and 'cr',
                          representing the counts of hits, misses, false alarms, and correct rejections
                          for each group.

  Returns:
      pd.DataFrame, List[str]: The original groupby-ed DataFrame with additional columns containing
                                SDT measures (d', beta, c, Ad, hit rate, miss rate, false alarm rate, and correct rejection rate).
                                The second returned value is a list of column names containing the calculated SDT measures
                                with the 'py_' prefix.

  Notes:
      - The function modifies the input DataFrame in place by adding the calculated SDT measures
        as new columns with 'py_' prefix (e.g., 'py_d', 'py_beta', 'py_c', 'py_Ad').
      - The 'calculate_sdt' function must be defined elsewhere to calculate the SDT measures for each row.
        Ensure that the 'calculate_sdt' function returns a dictionary containing the SDT measures as keys
        (e.g., {'d': d_value, 'beta': beta_value, 'c': c_value, 'Ad': Ad_value}).
  """
  sdt_meas = ['d', 'beta', 'c', 'Ad', 'hit_rate', 'miss_rate', 'fa_rate', 'cr_rate',]
  pysdt_meas = []
  for meas in sdt_meas:
      df[f'py_{meas}'] = df.apply(lambda x: calculate_sdt(x.hit, x.miss, x.fa, x.cr)[meas], axis=1)
      pysdt_meas.append(f'py_{meas}')
  return df, pysdt_meas

def add_stimlev_i(df):
  stimlev_i_cond = [
      (df['stimLev']==0),
      (df['stimLev']==0.06),
      (df['stimLev']==0.12),
      (df['stimLev']==0.24),
      (df['stimLev']==0.48),
      (df['stimLev']==8),
      (df['stimLev']==16),
      (df['stimLev']==32),
      (df['stimLev']==64),
  ]
  stimlev_i_val = [0, 1, 2, 3, 4, 1, 2, 3, 4]
  df['stimLev_i'] = np.select(stimlev_i_cond, stimlev_i_val)
  return df

def create_nR_s_arrays(df):
  stimTypes = {
      "s1": ["cr_4", "cr_3", "cr_2", "cr_1", "fa_1", "fa_2", "fa_3", "fa_4",],
      "s2": ["hit_4", "hit_3", "hit_2", "hit_1", "miss_1", "miss_2", "miss_3", "miss_4",]
  }

  for task in df.task.unique():
    for isStrength in df.isStrength.unique():
      for stimLev_i in df.stimLev_i.unique():
        for stimType in stimTypes:
          nR_s = df.query('task==@task & isStrength==@isStrength & stimLev_i==@stimLev_i')[stimTypes[stimType]].values
          if np.size(nR_s) > 0:
            np.savetxt(data_dir+f"/nR_s/{task}_grip{isStrength}_stim{stimLev_i}_nR_{stimType}.csv", nR_s, delimiter=',', fmt='%d')
            print(f"{task}_grip{isStrength}_stim{stimLev_i}_nR_{stimType}.csv")

def create_sub_task_stimlev_grip_nRs(df, subject_colname, data_dir, save='y', print_filepath='y'):
  """
  CRITICAL NOTES:
  This function prepares the confidence rating counts as inputs for meta-d or
  hmeta-d (MATLAB) packages, which use signal detection theory as their underlying
  framework. Trial types are generalized into either s1 (stimulus 1 presented) or
  s2 (stimulus 2 presented), and response types are similarly categorized as
  "s1" (responded s1) or "s2" (responded "s2").
  See link for details: https://github.com/metacoglab/HMeta-d/wiki/HMeta-d-tutorial#preparing-confidence-rating-data

  Within the context of auditory/visual discrimination task, s1, s2, and "s1", "s2"
  map onto the following:
  - correct rejections are "s1" to s1 (correct)
  - false alarms are "s2" to s1 (incorrect)
  - misses are "s1" to s2, hit is "s2" to s2 (incorrect)
  - hits are "s2" to s2 (correct)
  Therefore, the order of the confidence rating count columns need to look like this:
  s1: [cr_4, cr_3, cr_2, cr_1, fa_1, fa_2, fa_3, fa_4]
  s2: [miss_4, miss_3, miss_2, miss_1, hit_1, hit_2, hit_3, hit_4]

  side note: correct rejection and false alarm counts are repeated/propogated for
  each stimulus level via `propogate_fa_cr_stimlev`

  """

  stimTypes = {
      "s1": ["cr_4", "cr_3", "cr_2", "cr_1", "fa_1", "fa_2", "fa_3", "fa_4",],
      "s2": ["miss_4", "miss_3", "miss_2", "miss_1", "hit_1", "hit_2", "hit_3", "hit_4",],
  }

  # loop through all permutations of task (2: auditory/visual), grip (2: high/low), and stimulus level (4)
  for task, isStrength, stimLev_i in df[['task', 'isStrength', 'stimLev_i']].drop_duplicates().itertuples(index=False):

    # loop throgh the stimTypes dict to make a separate dataframe for each stimType s1 or s2 (standard or target)
    for stimType, column_names in stimTypes.items():
      # add the subject column in here as well
      nR_s = df[(df['task'] == task) & (df['isStrength'] == isStrength) & (df['stimLev_i'] == stimLev_i)][[subject_colname, *column_names]]

      # specify filename based on the iterators and whether it's an s1 or s2 (standard or target)
      filename = f"{task}_grip{isStrength}_stimlev{stimLev_i}_nR_{stimType}.csv"
      filepath = os.path.join(data_dir, "lc-hmetad", "lcya_nR_s", filename)

      # print out the name and size as sanity check
      if print_filepath=='y':
        print(filepath)

      print(f"{filename}: {len(nR_s)}")

      # save option
      if save=='y':
        nR_s.to_csv(filepath, index=False)

def create_sub_task_stimlev_nRs(df, subject_colname, data_dir, save='y', print_filepath='y'):
  """
  CRITICAL NOTES:
  This function prepares the confidence rating counts as inputs for meta-d or
  hmeta-d (MATLAB) packages, which use signal detection theory as their underlying
  framework. Trial types are generalized into either s1 (stimulus 1 presented) or
  s2 (stimulus 2 presented), and response types are similarly categorized as
  "s1" (responded s1) or "s2" (responded "s2").
  See link for details: https://github.com/metacoglab/HMeta-d/wiki/HMeta-d-tutorial#preparing-confidence-rating-data

  Within the context of auditory/visual discrimination task, s1, s2, and "s1", "s2"
  map onto the following:
  - correct rejections are "s1" to s1 (correct)
  - false alarms are "s2" to s1 (incorrect)
  - misses are "s1" to s2, hit is "s2" to s2 (incorrect)
  - hits are "s2" to s2 (correct)
  Therefore, the order of the confidence rating count columns need to look like this:
  s1: [cr_4, cr_3, cr_2, cr_1, fa_1, fa_2, fa_3, fa_4]
  s2: [miss_4, miss_3, miss_2, miss_1, hit_1, hit_2, hit_3, hit_4]

  side note: correct rejection and false alarm counts are repeated/propogated for
  each stimulus level via `propogate_fa_cr_stimlev`

  """

  stimTypes = {
      "s1": ["cr_4", "cr_3", "cr_2", "cr_1", "fa_1", "fa_2", "fa_3", "fa_4",],
      "s2": ["miss_4", "miss_3", "miss_2", "miss_1", "hit_1", "hit_2", "hit_3", "hit_4",],
  }

  # loop through all permutations of task (2: auditory/visual), grip (2: high/low), and stimulus level (4)
  for task, stimLev_i in df[['task', 'stimLev_i']].drop_duplicates().itertuples(index=False):

    # loop throgh the stimTypes dict to make a separate dataframe for each stimType s1 or s2 (standard or target)
    for stimType, column_names in stimTypes.items():
      # add the subject column in here as well
      nR_s = df[(df['task'] == task) & (df['stimLev_i'] == stimLev_i)][[subject_colname, *column_names]]

      # specify filename based on the iterators and whether it's an s1 or s2 (standard or target)
      filename = f"{task}_stimlev{stimLev_i}_nR_{stimType}.csv"
      filepath = os.path.join(data_dir, "lc-hmetad", "lcya_nR_s", filename)

      # print out the name and size as sanity check
      if print_filepath=='y':
        print(filepath)

      print(f"{filename}: {len(nR_s)}")

      # save option
      if save=='y':
        nR_s.to_csv(filepath, index=False)

# Function to calculate the py_d stimLev_i slopes for each combination of 'isStrength' and 'task'
def calculate_slopes(group):
    slopes = []
    for i in range(3):
        slope = (group['py_d'].iloc[i + 1] - group['py_d'].iloc[i]) / (group['stimLev_i'].iloc[i + 1] - group['stimLev_i'].iloc[i])
        slopes.append(slope)
    slopes.extend([float('nan')] * (len(group) - 3))  # Add NaNs for 'stimLev_i=4' and beyond
    group['py_d_slope_1_2'] = slopes[0]  # Slope from stimLev_i 1 to 2
    group['py_d_slope_2_3'] = slopes[1]  # Slope from stimLev_i 2 to 3
    group['py_d_slope_3_4'] = slopes[2]  # Slope from stimLev_i 3 to 4
    return group

def plot_psychometric_conf(df, plots, n_subjects):
  figsize = (10, 8)
  plt.figure(figsize=figsize)
  plt.rcParams['font.size'] = '16'

  x_lab = ["Standard", "8/0.6", "16/0.12", "32/0.24", "64/0.48"]
  x_pos = np.arange(len(x_lab))

  fig, ax = plt.subplots()
  ax2 = ax.twinx()

  for plot in plots:
      if 'hi' in plot:
          alpha=1
          isStrength=1
          grip='hi'
      elif 'lo' in plot:
          alpha=0.5
          isStrength=0
          grip='lo'
      if 'aud' in plot:
          color='b'
          task='aud'
      elif 'vis' in plot:
          color='r'
          task='vis'

      baralpha = alpha/3
      if 'hi' in plot:
          ax.bar(x_pos+0.2, df.query(f"task=='{task}' & isStrength=={isStrength}").reset_index().resp2, color=color, alpha=baralpha, width=0.3)
      elif 'lo' in plot:
          ax.bar(x_pos-0.2, df.query(f"task=='{task}' & isStrength=={isStrength}").reset_index().resp2, color=color, alpha=baralpha, width=0.3)

      ax2.scatter(x_pos[0], df.query(f"task=='{task}' & isStrength=={isStrength}").reset_index().iscorr[0], color=color, alpha=alpha)
      ax2.plot(x_pos[1:], df.query(f"task=='{task}' & isStrength=={isStrength}").reset_index().iscorr[1:], color=color, label=f'{task}, {grip} grip', alpha=alpha)

  ax2.set_ylim(0, 1)
  ax2.legend()
  ax2.set_xticks(x_pos)
  ax2.set_xticklabels(x_lab)
  ax2.set_ylabel("Mean Proportion Correct")

  ax.set_ylim(1, 4)
  ax.set_yticks([1, 2, 3, 4])
  ax.set_ylabel("Mean Confidence Rating")

  plt.title(f"N = {n_subjects}")

  fig.set_figheight(8)
  fig.set_figwidth(10)
  fig.show()



## Load error-tagged trial-level data
- `lcya_behdata_trial_cts_cleaned_tagged`

In [4]:
# set primary data directory
data_dir = os.path.join(mountpoint, 'Shareddrives/LC-Aging/Older Adult - MRI Study/Data Analysis/bap-behavioral')
# load trial-level data with the confusion matrix counts
trial_data = pd.read_csv(data_dir+'/lcya_behdata_trial_cts_cleaned_tagged.csv')
len(trial_data)

24000

In [5]:
%xmode Verbose

Exception reporting mode: Verbose


## Filters

In [6]:
[print(column) for column in trial_data.columns if 'invalid' in column or 'missed' in column]

missed_resp1
missed_resp2
invalid_resp1
invalid_grip
invalid_resp2
invalid_resp2_80
invalid_resp2_89
invalid_total


[None, None, None, None, None, None, None, None]

In [7]:
filter_missed_resp1 = trial_data.missed_resp1==0
filter_missed_resp2 = trial_data.missed_resp2==0
filter_invalid_resp1 = trial_data.invalid_resp1==0
filter_invalid_grip = trial_data.invalid_grip==0
# filter_invalid_resp2 = trial_data.invalid_resp2==0
# filter_invalid_resp2 = trial_data.invalid_resp2_80==0
filter_invalid_resp2 = trial_data.invalid_resp2_89==0

# D' and Slope

## Apply filters

In [8]:
trial_data_filtered = trial_data[
  filter_invalid_resp1
  & filter_invalid_grip
    ]
len(trial_data_filtered)

20478

## Group by subject, task, stimulus level, and grip level

In [9]:
groupby_list = ['sub', 'task', 'stimLev', 'isStrength']

sdt_cts_list = [
    'hit', 'miss', 'fa',  'cr',
    'hit_1', 'hit_2', 'hit_3', 'hit_4',
    'miss_1', 'miss_2', 'miss_3', 'miss_4',
    'fa_1', 'fa_2', 'fa_3', 'fa_4',
    'cr_1', 'cr_2', 'cr_3', 'cr_4',
]

# be aware of whether the df to groupby is fitered
sub_task_stimlev_grip = trial_data_filtered.groupby(groupby_list).agg({
    # first
    # "task":"first", "isStrength":"first",
    "stimLev_i":"first",
    # count
    "trial": "count",
    # sum (add counts)
    'hit':'sum', 'miss':'sum', 'fa':'sum', 'cr':'sum',
    'hit_1':'sum', 'hit_2':'sum','hit_3':'sum', 'hit_4':'sum',
    'miss_1':'sum', 'miss_2':'sum', 'miss_3':'sum', 'miss_4':'sum',
    'fa_1':'sum', 'fa_2':'sum', 'fa_3':'sum', 'fa_4':'sum',
    'cr_1':'sum', 'cr_2':'sum', 'cr_3':'sum', 'cr_4':'sum',
    # mean
    'resp1RT':'mean', 'resp2':'mean', 'resp2RT':'mean',
    'auc':'mean', 'auc_prop_targ':'mean', 'auc_rel_mvc':'mean',
    # 'py_d':'mean', 'py_beta':'mean', 'py_c':'mean', 'py_Ad':'mean',
    # 'py_hit_rate':'mean', 'py_miss_rate':'mean', 'py_fa_rate':'mean','py_cr_rate':'mean'
})

## Apply SDT functions (Propogate FA and CR, calculate D')

In [ ]:
sub_task_stimlev_grip_sdt = propogate_fa_cr_stimlev(sub_task_stimlev_grip, sdt_cts_list, groupby_list)
sub_task_stimlev_grip_sdt, pysdt_meas = calculate_sdt_df(sub_task_stimlev_grip_sdt)

## Slope of Psychometric Curve
- Unit-wise change in sensitivity (performance) as a function of change in stimulus strength

In [ ]:
len(sub_task_stimlev_grip_sdt['sub'].unique())

In [ ]:
data = sub_task_stimlev_grip_sdt
data = data.groupby(['sub', 'isStrength', 'task']).apply(calculate_slopes).reset_index()
data.groupby(['isStrength' , 'task']).mean().reset_index()[['isStrength' , 'task', 'py_d', 'py_d_slope_1_2',	'py_d_slope_2_3',	'py_d_slope_3_4',]].to_csv(data_dir + '/BAP_grip_task_slope.csv')

## Save output

In [ ]:
# save trial-level data with confusion matrix counts as csv
save_csv = input('Save CSV? (y/n): ')
if save_csv=='y':
  data.to_csv(data_dir+'/lcya_sub_task_stimlev_grip_sdt.csv', index=None)

# Meta-D: by subject, task, stimulus level, and grip

## Apply filters

In [ ]:
trial_data_filtered = trial_data[
    filter_missed_resp1
    & filter_invalid_resp1
    & filter_missed_resp2
    & filter_invalid_grip
    & filter_invalid_resp2
    ]
len(trial_data_filtered)

## Group by subject, task, stimulus level, and grip

In [ ]:
groupby_list = ['sub', 'task', 'stimLev', 'isStrength']

sdt_cts_list = [
    'hit', 'miss', 'fa',  'cr',
    'hit_1', 'hit_2', 'hit_3', 'hit_4',
    'miss_1', 'miss_2', 'miss_3', 'miss_4',
    'fa_1', 'fa_2', 'fa_3', 'fa_4',
    'cr_1', 'cr_2', 'cr_3', 'cr_4',
]

# be aware of whether the df to groupby is fitered
sub_task_stimlev_grip = trial_data_filtered.groupby(groupby_list).agg({
    # first
    # "task":"first", "isStrength":"first",
    "stimLev_i":"first",
    # count
    "trial": "count",
    # sum (add counts)
    'hit':'sum', 'miss':'sum', 'fa':'sum', 'cr':'sum',
    'hit_1':'sum', 'hit_2':'sum','hit_3':'sum', 'hit_4':'sum',
    'miss_1':'sum', 'miss_2':'sum', 'miss_3':'sum', 'miss_4':'sum',
    'fa_1':'sum', 'fa_2':'sum', 'fa_3':'sum', 'fa_4':'sum',
    'cr_1':'sum', 'cr_2':'sum', 'cr_3':'sum', 'cr_4':'sum',
    # mean
    'resp1RT':'mean', 'resp2':'mean', 'resp2RT':'mean',
    'auc':'mean', 'auc_prop_targ':'mean', 'auc_rel_mvc':'mean',
})

## Apply SDT functions (Propogate FA and CR, calculate D')
- HMetaD Package also returns D', which can be compared against Python function.

In [ ]:
sub_task_stimlev_grip_sdt = propogate_fa_cr_stimlev(sub_task_stimlev_grip, sdt_cts_list, groupby_list)
sub_task_stimlev_grip_sdt, pysdt_meas = calculate_sdt_df(sub_task_stimlev_grip_sdt)

## Create outputs for HmetaD (MATLAB)

### adding 'sub-' prefix for subject column
the hmeta-d live script I'm using currently expected a string for this column

In [ ]:
sub_task_stimlev_grip_sdt['subj_id'] = 'sub-' + sub_task_stimlev_grip_sdt['sub'].astype(str)
sub_task_stimlev_grip_sdt['subj_id']

### temporary save of the groupbyed data

In [ ]:
data_dir = os.path.join(mountpoint, 'Shareddrives/LC-Aging/Older Adult - MRI Study/Data Analysis/bap-behavioral')
sub_task_stimlev_grip_sdt.to_csv(os.path.join(data_dir, 'lcya_sub_task_stimlev_grip_sdt.csv'), index=None)

### create nRs arrays

In [ ]:
df = sub_task_stimlev_grip_sdt
create_sub_task_stimlev_grip_nRs(df=df, subject_colname = 'subj_id', data_dir=data_dir, save='y', print_filepath='y')

# Meta-D: by subject, task, stimulus level (no grip)

> Indented block



## Apply filters

In [ ]:
trial_data_filtered = trial_data[
    filter_missed_resp1
    # & filter_invalid_resp1
    & filter_missed_resp2
    # & filter_invalid_grip
    & filter_invalid_resp2
    ]
len(trial_data_filtered)

## Group by subject, task, stimulus level (no grip)

In [ ]:
groupby_list = ['sub', 'task', 'stimLev', ]

sdt_cts_list = [
    'hit', 'miss', 'fa',  'cr',
    'hit_1', 'hit_2', 'hit_3', 'hit_4',
    'miss_1', 'miss_2', 'miss_3', 'miss_4',
    'fa_1', 'fa_2', 'fa_3', 'fa_4',
    'cr_1', 'cr_2', 'cr_3', 'cr_4',
]

# be aware of whether the df to groupby is fitered
sub_task_stimlev = trial_data_filtered.groupby(groupby_list).agg({
    # first
    # "task":"first",
    "stimLev_i":"first",
    # count
    "trial": "count",
    # sum (add counts)
    'hit':'sum', 'miss':'sum', 'fa':'sum', 'cr':'sum',
    'hit_1':'sum', 'hit_2':'sum','hit_3':'sum', 'hit_4':'sum',
    'miss_1':'sum', 'miss_2':'sum', 'miss_3':'sum', 'miss_4':'sum',
    'fa_1':'sum', 'fa_2':'sum', 'fa_3':'sum', 'fa_4':'sum',
    'cr_1':'sum', 'cr_2':'sum', 'cr_3':'sum', 'cr_4':'sum',
    # mean
    'resp1RT':'mean', 'resp2':'mean', 'resp2RT':'mean',
    'auc':'mean', 'auc_prop_targ':'mean', 'auc_rel_mvc':'mean',
})

sub_task_stimlev

## Apply SDT functions

In [ ]:
sub_task_stimlev_sdt = propogate_fa_cr_stimlev(sub_task_stimlev, sdt_cts_list, groupby_list)
sub_task_stimlev_sdt, pysdt_meas = calculate_sdt_df(sub_task_stimlev_sdt)

## Create outputs for HmetaD (MATLAB)

### adding 'sub-' prefix for subject column
the hmeta-d live script I'm using currently expected a string for this column

In [ ]:
sub_task_stimlev_sdt['subj_id'] = 'sub-' + sub_task_stimlev_sdt['sub'].astype(str)
sub_task_stimlev_sdt['subj_id']

### temporary save of the groupbyed data

In [ ]:
data_dir = os.path.join(mountpoint, 'Shareddrives/LC-Aging/Older Adult - MRI Study/Data Analysis/bap-behavioral')
sub_task_stimlev_sdt.to_csv(os.path.join(data_dir, 'lcya_sub_task_stimlev_sdt.csv'), index=None)

### create nRs arrays

In [ ]:
create_sub_task_stimlev_nRs(df=sub_task_stimlev_sdt, subject_colname = 'subj_id', data_dir=data_dir, save='y', print_filepath='y')